# Final System — All 3 Tasks + Model Save/Load + Inference

**Best architecture from ablation: V3-D (No Gating)**
- Removes TokenLevelGating, keeps CrossAttention + GraphReasoning + ClausePool
- Uses clause-level cross-attention for question-answer alignment instead of token-level gate

**Three tasks:**
1. **Task 1 — Evasion Detection** (3-class: Non-Evasive / Partially Evasive / Evasive)
2. **Task 2 — Stance Classification** (binary: FAVOR / AGAINST)
3. **Task 3 — Explanation Span Extraction** (which tokens/clauses caused the evasion prediction)

**Model saving:** best checkpoint saved per task to `/kaggle/working/`
**Inference notebook:** separate section loads saved models and runs on new examples

In [1]:
!pip install transformers scikit-learn pandas torch tqdm --quiet
print("Libraries installed.")

Libraries installed.


In [2]:
import os, re, random, warnings, itertools
import pandas as pd, numpy as np
import torch, torch.nn as nn
import torch.nn.functional as F
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertModel, DistilBertTokenizer, get_cosine_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, classification_report, confusion_matrix)
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
from IPython.display import display, HTML
import time
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda": print(f"GPU: {torch.cuda.get_device_name(0)}")

DATA_DIR   = "/kaggle/input/datasets/varsha0606/inlp-datasets"
OUTPUT_DIR = "/kaggle/working"

BERT_MODEL     = "distilbert-base-uncased"
MAX_LEN        = 256
STANCE_MAX_LEN = 128
Q_MAX_LEN      = 64
MAX_CLAUSES    = 6
CLAUSE_LEN     = 64
BATCH_SIZE     = 16
GRAD_ACCUM     = 2
EPOCHS         = 5
PATIENCE       = 3
DROPOUT        = 0.3
WARMUP_RATIO   = 0.1
FREEZE_EPOCHS  = 2
LR_EVASION     = 1e-5
LR_STANCE      = 2e-5
LABEL_SMOOTH   = 0.1

EVASION_LABELS   = ["Non-Evasive", "Partially Evasive", "Evasive"]
EVASION_LABEL2ID = {l: i for i, l in enumerate(EVASION_LABELS)}
STANCE_LABELS    = ["FAVOR", "AGAINST"]
STANCE_LABEL2ID  = {l: i for i, l in enumerate(STANCE_LABELS)}

EVASION_CSV       = f"{DATA_DIR}/QAEvasion.csv"
STANCE_TRAIN_CSVS = [f"{DATA_DIR}/raw_train_biden.csv",f"{DATA_DIR}/raw_train_trump.csv",f"{DATA_DIR}/raw_train_bernie.csv"]
STANCE_VAL_CSVS   = [f"{DATA_DIR}/raw_val_biden.csv",  f"{DATA_DIR}/raw_val_trump.csv",  f"{DATA_DIR}/raw_val_bernie.csv"]
STANCE_TEST_CSVS  = [f"{DATA_DIR}/raw_test_biden.csv", f"{DATA_DIR}/raw_test_trump.csv", f"{DATA_DIR}/raw_test_bernie.csv"]

# Model save paths
EVASION_MODEL_PATH = os.path.join(OUTPUT_DIR, "best_model_evasion.pt")
STANCE_MODEL_PATH  = os.path.join(OUTPUT_DIR, "best_model_stance.pt")
FULL_MODEL_PATH    = os.path.join(OUTPUT_DIR, "best_model_full.pt")

print("Config loaded.")
print(f"  Architecture: V3-D (No Gating — CrossAttn + Graph + ClausePool)")
print(f"  Batch:{BATCH_SIZE}  GradAccum:{GRAD_ACCUM}  EffBatch:{BATCH_SIZE*GRAD_ACCUM}")
print(f"  Epochs:{EPOCHS}  Patience:{PATIENCE}")

Device: cuda
GPU: Tesla T4
Config loaded.
  Architecture: V3-D (No Gating — CrossAttn + Graph + ClausePool)
  Batch:16  GradAccum:2  EffBatch:32
  Epochs:5  Patience:3


In [3]:
# ─────────────────────────────────────────────────────────────
# AUGMENTATION — 5 strategies on PE class only (8x)
# ─────────────────────────────────────────────────────────────

SYNONYM_MAP = {
    "question":["query","inquiry"],"answer":["response","reply"],
    "policy":["plan","strategy"],"government":["administration","authority"],
    "issue":["matter","concern","problem"],"important":["crucial","significant","vital"],
    "believe":["think","consider","feel"],"people":["citizens","individuals","public"],
    "country":["nation","state"],"support":["back","endorse","advocate"],
    "change":["shift","reform","alter"],"need":["require","must"],
    "work":["function","operate","effort"],"make":["create","produce","form"],
    "good":["beneficial","positive","effective"],"said":["stated","noted","mentioned"],
    "know":["understand","recognize","realize"],"think":["believe","consider","feel"],
    "want":["desire","seek","aim"],"time":["period","moment","point"],
    "new":["recent","fresh","updated"],"political":["governmental","civic"],
    "major":["significant","key","primary"],"different":["various","distinct","alternative"],
    "president":["commander-in-chief","leader","executive"],
    "congress":["legislature","lawmakers","senate"],
    "economy":["market","financial system","fiscal situation"],
    "certainly":["absolutely","definitely","of course"],
    "perhaps":["possibly","maybe","one could argue"],
    "however":["that said","nevertheless","on the other hand"],
    "basically":["fundamentally","essentially","at its core"],
}

EVASION_PHRASES = {
    "i think we should":"it is my view that we ought to",
    "i believe that":"it is my conviction that",
    "we need to":"it is essential that we",
    "i want to":"my intention is to",
    "we are working on":"efforts are underway to",
    "i don't think":"it is not my view that",
    "the fact is":"what we know is",
    "i would say":"my position would be",
    "let me be clear":"to make this absolutely clear",
    "the truth is":"the reality of the situation is",
    "we have to":"it is necessary that we",
    "i am committed to":"my commitment remains to",
    "going forward":"in the coming period",
    "at the end of the day":"ultimately",
    "we must":"it is imperative that we",
    "i am confident":"i have full confidence",
}

STOPWORDS = {"the","a","an","is","are","was","were","be","been","being",
             "have","has","had","do","does","did","will","would","could",
             "should","may","might","to","of","in","for","on","with",
             "at","by","from","as","into","i","we","it","this","that"}

def aug_synonym_replace(text, n=3, seed=None):
    if seed is not None: random.seed(seed)
    words = text.split()
    rep = [(i,w.lower().strip('.,!?;:"\'()')) for i,w in enumerate(words)
           if w.lower().strip('.,!?;:"\'()') in SYNONYM_MAP]
    if not rep: return text
    chosen = random.sample(rep, min(n, len(rep)))
    nw = words[:]
    for i,w in chosen:
        syn = random.choice(SYNONYM_MAP[w])
        if words[i][0].isupper(): syn=syn[0].upper()+syn[1:]
        nw[i] = syn
    return " ".join(nw)

def aug_structure_swap(text, n=2, seed=None):
    if seed is not None: random.seed(seed)
    clauses = re.split(r'(?<=[.!?,;])\s+', text)
    clauses = [c for c in clauses if len(c.split()) >= 4]
    if not clauses:
        words = text.split()
        if len(words) < 4: return text
        for _ in range(n):
            i,j = random.sample(range(len(words)),2)
            words[i],words[j] = words[j],words[i]
        return " ".join(words)
    nc = []
    for clause in clauses:
        words = clause.split()
        if len(words) >= 4 and random.random() < 0.6:
            for _ in range(min(n, len(words)//2)):
                i,j = random.sample(range(len(words)),2)
                words[i],words[j] = words[j],words[i]
        nc.append(" ".join(words))
    return " ".join(nc)

def aug_conservative_deletion(text, p=0.07, seed=None):
    if seed is not None: random.seed(seed)
    words = text.split()
    if len(words) <= 6: return text
    nw = []
    for w in words:
        clean = w.lower().strip('.,!?;:"\'()')
        if clean in STOPWORDS or len(clean)<=2: nw.append(w)
        elif random.random()>p: nw.append(w)
    return " ".join(nw) if len(nw)>=len(words)//2 else text

def aug_phrase_paraphrase(text, seed=None):
    if seed is not None: random.seed(seed)
    tl = text.lower()
    cands = [(p,r) for p,r in EVASION_PHRASES.items() if p in tl]
    if not cands: return aug_synonym_replace(text, n=2, seed=seed)
    chosen = random.sample(cands, min(random.randint(1,2), len(cands)))
    result = text
    for phrase,replacement in chosen:
        idx = result.lower().find(phrase)
        if idx >= 0:
            if idx==0 or result[idx-1] in '.!?':
                replacement=replacement[0].upper()+replacement[1:]
            result=result[:idx]+replacement+result[idx+len(phrase):]
    return result

def aug_combined_chain(text, seed=None):
    if seed is not None: random.seed(seed)
    return aug_structure_swap(aug_synonym_replace(text,n=2,seed=seed),n=1,seed=seed+1)

AUG_OPS = [aug_synonym_replace,aug_structure_swap,aug_conservative_deletion,
           aug_phrase_paraphrase,aug_combined_chain]

def augment_pe_samples(questions, answers, labels, target_label_id=1,
                       augment_factor=8, seed=42):
    aug_q,aug_a,aug_l = [],[],[]
    pe_idx = [i for i,l in enumerate(labels) if l==target_label_id]
    print(f"  PE samples:{len(pe_idx)}  factor:{augment_factor}x  -> +{len(pe_idx)*augment_factor} samples")
    for i,idx in enumerate(pe_idx):
        q,a,l = questions[idx],answers[idx],labels[idx]
        for k in range(augment_factor):
            op = AUG_OPS[k%len(AUG_OPS)]
            aug_q.append(q); aug_a.append(op(a,seed+i*500+k)); aug_l.append(l)
    return aug_q,aug_a,aug_l

print("Augmentation module ready.")

Augmentation module ready.


In [4]:
def map_evasion_label(raw):
    raw = str(raw).strip()
    if raw.startswith("1."):               return "Non-Evasive"
    elif raw == "2.3 Partial/half-answer": return "Partially Evasive"
    else:                                  return "Evasive"

def load_evasion(filepath):
    print("Loading QA Evasion...")
    df = pd.read_csv(filepath)
    df["question"] = df["interview_question"].fillna("").str.strip()
    df["answer"]   = df["interview_answer"].fillna("").str.strip()
    df["coarse_label"] = df["label"].apply(map_evasion_label)
    df["label_id"]     = df["coarse_label"].map(EVASION_LABEL2ID).astype(int)
    df = df.dropna(subset=["question","answer","label_id"]).reset_index(drop=True)
    print(f"  Total:{len(df)}")
    for lbl in EVASION_LABELS:
        n=(df["coarse_label"]==lbl).sum()
        print(f"  {lbl:22s}: {n:4d} ({n/len(df)*100:.1f}%)")
    idx=list(range(len(df))); lbl=df["label_id"].tolist()
    itr,itmp,_,ytmp = train_test_split(idx,lbl,test_size=0.20,random_state=SEED,stratify=lbl)
    iva,ite,_,_     = train_test_split(itmp,ytmp,test_size=0.50,random_state=SEED,stratify=ytmp)
    q,a,l = df["question"].tolist(),df["answer"].tolist(),df["label_id"].tolist()
    def ex(ii): return [q[i] for i in ii],[a[i] for i in ii],[l[i] for i in ii]
    tr,va,te = ex(itr),ex(iva),ex(ite)
    print(f"  Train:{len(itr)}  Val:{len(iva)}  Test:{len(ite)}")
    return tr,va,te

def load_stance_split(filepaths, split_name):
    dfs=[]
    for fp in filepaths:
        if os.path.exists(fp): dfs.append(pd.read_csv(fp))
    df=pd.concat(dfs,ignore_index=True)
    df=df[df["Stance"].isin(STANCE_LABEL2ID)].reset_index(drop=True)
    df["label_id"]=df["Stance"].map(STANCE_LABEL2ID).astype(int)
    df["target"]=df["Target"].fillna("").str.strip()
    df["tweet"]=df["Tweet"].fillna("").str.strip()
    df=df.dropna(subset=["target","tweet","label_id"]).reset_index(drop=True)
    return df["target"].tolist(),df["tweet"].tolist(),df["label_id"].tolist()

print("="*60)
(Qetr,Aetr,yetr),(Qeva,Aeva,yeva),(Qete,Aete,yete) = load_evasion(EVASION_CSV)

print()
print("Augmenting PE class in training set (8x)...")
aug_q,aug_a,aug_l = augment_pe_samples(Qetr,Aetr,yetr,target_label_id=1,augment_factor=8)
Qetr_aug=Qetr+aug_q; Aetr_aug=Aetr+aug_a; yetr_aug=yetr+aug_l
combined=list(zip(Qetr_aug,Aetr_aug,yetr_aug))
random.seed(SEED); random.shuffle(combined)
Qetr_aug,Aetr_aug,yetr_aug=[list(x) for x in zip(*combined)]
print(f"  Augmented train:{len(yetr_aug)}")
for lid,lbl in enumerate(EVASION_LABELS):
    n=sum(1 for y in yetr_aug if y==lid)
    print(f"  {lbl:22s}: {n:4d} ({n/len(yetr_aug)*100:.1f}%)")

print()
Qstr,Astr,ystr = load_stance_split(STANCE_TRAIN_CSVS,"train")
Qsva,Asva,ysva = load_stance_split(STANCE_VAL_CSVS,"val")
Qste,Aste,yste = load_stance_split(STANCE_TEST_CSVS,"test")
print(f"Stance -> train:{len(ystr)}  val:{len(ysva)}  test:{len(yste)}")
print("="*60)

ew = torch.tensor(compute_class_weight("balanced",classes=np.array([0,1,2]),y=yetr_aug),dtype=torch.float).to(device)
sw = torch.tensor(compute_class_weight("balanced",classes=np.array([0,1]),  y=ystr),dtype=torch.float).to(device)
print(f"Evasion weights: {[round(x,4) for x in ew.tolist()]}")
print(f"Stance  weights: {[round(x,4) for x in sw.tolist()]}")

Loading QA Evasion...
  Total:3448
  Non-Evasive           : 1540 (44.7%)
  Partially Evasive     :   79 (2.3%)
  Evasive               : 1829 (53.0%)
  Train:2758  Val:345  Test:345

Augmenting PE class in training set (8x)...
  PE samples:63  factor:8x  -> +504 samples
  Augmented train:3262
  Non-Evasive           : 1232 (37.8%)
  Partially Evasive     :  567 (17.4%)
  Evasive               : 1463 (44.8%)

Stance -> train:17224  val:2193  test:2157
Evasion weights: [0.8826, 1.9177, 0.7432]
Stance  weights: [1.0317, 0.9701]


In [5]:
tokenizer = DistilBertTokenizer.from_pretrained(BERT_MODEL)

def split_into_clauses(text):
    parts = re.split(
        r'(?<=[.!?])\s+|(?:\s+(?:but|however|although|whereas|while|yet|'
        r'because|since|so|therefore|nevertheless|nonetheless)\s+)', text)
    parts = [p.strip() for p in parts if len(p.strip()) > 5]
    return parts[:MAX_CLAUSES] if parts else [text[:500]]

class TaskDataset(Dataset):
    def __init__(self, ta, tb, labels, max_len, task_id):
        self.ta,self.tb,self.labels,self.max_len,self.task_id=ta,tb,labels,max_len,task_id
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        enc  = tokenizer(self.ta[idx],self.tb[idx],truncation=True,
                         padding="max_length",max_length=self.max_len,return_tensors="pt")
        qenc = tokenizer(self.ta[idx],truncation=True,
                         padding="max_length",max_length=Q_MAX_LEN,return_tensors="pt")
        clauses = split_into_clauses(self.tb[idx]) if self.task_id==0 else [self.tb[idx]]
        ids_l,masks_l=[],[]
        for c in clauses[:MAX_CLAUSES]:
            ce=tokenizer(self.ta[idx],c,truncation=True,
                         padding="max_length",max_length=CLAUSE_LEN,return_tensors="pt")
            ids_l.append(ce["input_ids"].squeeze(0))
            masks_l.append(ce["attention_mask"].squeeze(0))
        n_real=len(ids_l)
        while len(ids_l)<MAX_CLAUSES:
            ids_l.append(torch.zeros(CLAUSE_LEN,dtype=torch.long))
            masks_l.append(torch.zeros(CLAUSE_LEN,dtype=torch.long))
        return {
            "input_ids":       enc["input_ids"].squeeze(0),
            "attention_mask":  enc["attention_mask"].squeeze(0),
            "q_input_ids":     qenc["input_ids"].squeeze(0),
            "q_attention_mask":qenc["attention_mask"].squeeze(0),
            "clause_ids":      torch.stack(ids_l),
            "clause_masks":    torch.stack(masks_l),
            "n_clauses":       torch.tensor(n_real,dtype=torch.long),
            "labels":          torch.tensor(self.labels[idx],dtype=torch.long),
            "task_id":         torch.tensor(self.task_id,dtype=torch.long),
        }

def make_loader(ta,tb,y,tid,ml,shuffle):
    return DataLoader(TaskDataset(ta,tb,y,ml,tid),batch_size=BATCH_SIZE,
                      shuffle=shuffle,num_workers=2,pin_memory=True)

loaders = {
    "evasion_train": make_loader(Qetr_aug,Aetr_aug,yetr_aug,0,MAX_LEN,True),
    "evasion_val":   make_loader(Qeva,Aeva,yeva,0,MAX_LEN,False),
    "evasion_test":  make_loader(Qete,Aete,yete,0,MAX_LEN,False),
    "stance_train":  make_loader(Qstr,Astr,ystr,1,STANCE_MAX_LEN,True),
    "stance_val":    make_loader(Qsva,Asva,ysva,1,STANCE_MAX_LEN,False),
    "stance_test":   make_loader(Qste,Aste,yste,1,STANCE_MAX_LEN,False),
}
for k,v in loaders.items():
    print(f"  {k:<20} {len(v.dataset):>7,} samples  {len(v):>4} batches")
print("DataLoaders ready.")

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

  evasion_train          3,262 samples   204 batches
  evasion_val              345 samples    22 batches
  evasion_test             345 samples    22 batches
  stance_train          17,224 samples  1077 batches
  stance_val             2,193 samples   138 batches
  stance_test            2,157 samples   135 batches
DataLoaders ready.


In [6]:
def mean_pooling(h, mask):
    m = mask.unsqueeze(-1).expand(h.size()).float()
    return torch.sum(h*m,1)/torch.clamp(m.sum(1),min=1e-9)

class CrossAttention(nn.Module):
    """
    Question attends over clause representations.
    Returns a single question-weighted summary of which clauses
    are most relevant to the question — the core alignment signal.
    """
    def __init__(self, H):
        super().__init__()
        self.Wq=nn.Linear(H,H); self.Wk=nn.Linear(H,H); self.Wv=nn.Linear(H,H)
    def forward(self, q, cr, n_clauses):
        B,C,H=cr.size()
        scale=torch.sqrt(torch.tensor(H,dtype=torch.float32,device=q.device))
        scores=torch.bmm(self.Wq(q).unsqueeze(1),self.Wk(cr).transpose(1,2))/scale
        valid=torch.arange(C,device=q.device).unsqueeze(0)<n_clauses.unsqueeze(1)
        scores=scores.squeeze(1).masked_fill(~valid,float("-inf"))
        w=F.softmax(scores,dim=-1); w=w/(w.sum(-1,keepdim=True)+1e-9)
        w=F.dropout(w,p=0.1,training=self.training)
        # Save attention weights for Task 3 explanation extraction
        self.last_attn_weights = w.detach()
        return torch.bmm(w.unsqueeze(1),self.Wv(cr)).squeeze(1)

class GraphReasoningLayer(nn.Module):
    """
    2-round message passing over clause nodes.
    Propagates evasion signals across the full answer.
    """
    def __init__(self, H, dr, rounds=2):
        super().__init__()
        self.layers=nn.ModuleList([nn.Sequential(nn.Linear(H*2,H),nn.ReLU(),nn.Dropout(dr))
                                   for _ in range(rounds)])
        self.norm=nn.LayerNorm(H)
    def forward(self, x, n_clauses):
        B,C,H=x.size()
        valid=(torch.arange(C,device=x.device).unsqueeze(0)<n_clauses.unsqueeze(1)).float()
        for layer in self.layers:
            sx=(x*valid.unsqueeze(-1)).sum(1,keepdim=True)
            denom=(n_clauses.unsqueeze(-1).unsqueeze(-1)-1).clamp(min=1)
            nbr=(sx-x*valid.unsqueeze(-1))/denom
            x=self.norm(x+layer(torch.cat([x,nbr],dim=-1))*valid.unsqueeze(-1))
        return x

class ClauseAttentionPooling(nn.Module):
    """
    Attention-weighted pooling over graph-updated clause representations.
    Learns an importance scalar per clause.
    """
    def __init__(self, H, dr):
        super().__init__()
        self.score=nn.Sequential(nn.Linear(H,H),nn.Tanh(),nn.Linear(H,1))
        self.norm=nn.LayerNorm(H); self.drop=nn.Dropout(dr)
    def forward(self, x, n_clauses):
        valid=(torch.arange(x.size(1),device=x.device).unsqueeze(0)<n_clauses.unsqueeze(1))
        s=self.score(x).squeeze(-1).masked_fill(~valid,float("-inf"))
        w=F.softmax(s,dim=-1)
        # Save clause importance for Task 3
        self.last_clause_weights = w.detach()
        w=w.unsqueeze(-1)
        return self.norm(self.drop((w*x).sum(1)))

class ClassHead(nn.Module):
    def __init__(self, H, n, dr):
        super().__init__()
        self.net=nn.Sequential(nn.Dropout(dr),nn.Linear(H,256),nn.ReLU(),
                               nn.Dropout(dr),nn.Linear(256,n))
    def forward(self, x): return self.net(x)

class MultiTaskNoGating(nn.Module):
    """
    V3-D: No Gating architecture.
    Pipeline: DistilBERT -> mean pool -> CrossAttention over clauses
              -> GraphReasoning -> ClauseAttentionPooling -> Fusion -> ClassHead

    Question-answer alignment is handled by CrossAttention at the clause level
    instead of token-level gating. This is architecturally cleaner and
    cross-attention weights are directly interpretable for Task 3.

    Task 3 explanation: cross_attn.last_attn_weights gives which clauses
    the question focused on, and cl_pool.last_clause_weights gives which
    clauses were most important after graph reasoning.
    """
    def __init__(self, ew, sw):
        super().__init__()
        self.encoder    = DistilBertModel.from_pretrained(BERT_MODEL)
        H = self.encoder.config.hidden_size
        self.clause_proj = nn.Linear(H,H)
        self.cross_attn  = CrossAttention(H)
        self.graph       = GraphReasoningLayer(H,DROPOUT,rounds=2)
        self.cl_pool     = ClauseAttentionPooling(H,DROPOUT)
        self.fusion      = nn.LayerNorm(H)
        self.evasion_head = ClassHead(H,len(EVASION_LABELS),DROPOUT)
        self.stance_head  = ClassHead(H,len(STANCE_LABELS), DROPOUT)
        self.evasion_loss_fn = nn.CrossEntropyLoss(weight=ew,label_smoothing=LABEL_SMOOTH)
        self.stance_loss_fn  = nn.CrossEntropyLoss(weight=sw,label_smoothing=LABEL_SMOOTH)

    def _encode_clauses(self, cids, cmasks, B, C):
        ids=cids.view(B*C,-1); masks=cmasks.view(B*C,-1)
        valid=masks.sum(dim=1)>0
        if valid.sum()==0:
            return torch.zeros(B,C,self.encoder.config.hidden_size,device=ids.device)
        vh=self.encoder(input_ids=ids[valid],attention_mask=masks[valid]).last_hidden_state
        p=mean_pooling(vh,masks[valid])
        pool=torch.zeros(ids.size(0),p.size(-1),device=ids.device); pool[valid]=p
        return F.relu(self.clause_proj(pool)).view(B,C,-1)

    def freeze_inactive_head(self, task):
        off = self.stance_head  if task==0 else self.evasion_head
        on  = self.evasion_head if task==0 else self.stance_head
        for p in off.parameters(): p.requires_grad=False
        for m in [on,self.encoder,self.clause_proj,self.cross_attn,
                  self.graph,self.cl_pool,self.fusion]:
            for p in m.parameters(): p.requires_grad=True

    def unfreeze_all(self):
        for p in self.parameters(): p.requires_grad=True

    def forward(self, input_ids, attention_mask, task_id, labels=None,
                q_input_ids=None, q_attention_mask=None,
                clause_ids=None, clause_masks=None, n_clauses=None):
        B=input_ids.size(0); task=int(task_id[0].item())

        # Global encoding
        h =self.encoder(input_ids=input_ids,attention_mask=attention_mask).last_hidden_state
        qh=self.encoder(input_ids=q_input_ids,attention_mask=q_attention_mask).last_hidden_state
        q_rep=F.normalize(mean_pooling(qh,q_attention_mask),dim=-1)

        # No gating — base is plain mean pool of answer
        base = mean_pooling(h,attention_mask)

        # Clause pipeline
        C=clause_ids.size(1)
        cr=self._encode_clauses(clause_ids,clause_masks,B,C)
        # CrossAttention: q_rep attends over clauses (interpretable for Task 3)
        xa=self.cross_attn(q_rep,cr,n_clauses)
        # Graph: propagate signals across clauses
        cr=self.graph(cr,n_clauses)
        # ClausePool: weighted sum (weights saved for Task 3)
        cl=self.cl_pool(cr,n_clauses)

        # Fusion: global + cross-attn + clause-pool
        fused=self.fusion(F.dropout(base+0.7*xa+0.7*cl,p=0.2,training=self.training))
        logits=(self.evasion_head if task==0 else self.stance_head)(fused)
        loss=((self.evasion_loss_fn if task==0 else self.stance_loss_fn)(logits,labels)
              if labels is not None else None)
        return loss,logits

print("Model defined: MultiTaskNoGating (V3-D)")
print("  Architecture: MeanPool + CrossAttention + GraphReasoning + ClausePool")
print("  cross_attn.last_attn_weights -> clause relevance (Task 3)")
print("  cl_pool.last_clause_weights  -> clause importance (Task 3)")

Model defined: MultiTaskNoGating (V3-D)
  Architecture: MeanPool + CrossAttention + GraphReasoning + ClausePool
  cross_attn.last_attn_weights -> clause relevance (Task 3)
  cl_pool.last_clause_weights  -> clause importance (Task 3)


In [7]:
from tqdm.auto import tqdm

@torch.no_grad()
def evaluate(model, loader, label_names):
    model.eval(); model.unfreeze_all()
    all_preds,all_labels,total_loss=[],[],0.0
    for batch in loader:
        loss,logits=model(
            batch["input_ids"].to(device),batch["attention_mask"].to(device),
            batch["task_id"].to(device),  batch["labels"].to(device),
            batch["q_input_ids"].to(device),batch["q_attention_mask"].to(device),
            batch["clause_ids"].to(device),batch["clause_masks"].to(device),
            batch["n_clauses"].to(device))
        total_loss+=loss.item()
        all_preds.extend(torch.argmax(logits,1).cpu().tolist())
        all_labels.extend(batch["labels"].tolist())
    return {
        "loss":  total_loss/len(loader),
        "acc":   accuracy_score(all_labels,all_preds),
        "f1":    f1_score(all_labels,all_preds,average="macro",zero_division=0),
        "prec":  precision_score(all_labels,all_preds,average="macro",zero_division=0),
        "rec":   recall_score(all_labels,all_preds,average="macro",zero_division=0),
        "report":classification_report(all_labels,all_preds,target_names=label_names,zero_division=0),
        "cm":    pd.DataFrame(
                   confusion_matrix(all_labels,all_preds,labels=list(range(len(label_names)))),
                   index=[f"True:{l}" for l in label_names],
                   columns=[f"Pred:{l}" for l in label_names]),
        "preds":all_preds,"labels":all_labels,
    }

def print_eval(m, title):
    print(f"\n{'='*60}\n  {title}\n{'='*60}")
    print(f"  Loss:{m['loss']:.4f}  Acc:{m['acc']*100:.2f}%  MacroF1:{m['f1']:.4f}  "
          f"Prec:{m['prec']:.4f}  Rec:{m['rec']:.4f}")
    print("\n  Per-class Report:")
    for line in m["report"].strip().split("\n"): print(f"  {line}")
    print("\n  Confusion Matrix:")
    print(m["cm"].to_string()); print("="*60)

def get_pe_f1(report_str):
    for line in report_str.split("\n"):
        if "Partially Evasive" in line:
            try: return float(line.split()[3])
            except: return 0.0
    return 0.0

def train_and_save(model, save_path):
    best_avg_f1=0.0; patience_ctr=0

    shared=list(model.encoder.parameters())+list(model.clause_proj.parameters())
    shared+=list(model.cross_attn.parameters())+list(model.graph.parameters())
    shared+=list(model.cl_pool.parameters())+list(model.fusion.parameters())
    ev_task=list(model.evasion_head.parameters())
    st_task=list(model.stance_head.parameters())

    optimizer=AdamW([{"params":ev_task,"lr":LR_EVASION},
                     {"params":st_task,"lr":LR_STANCE},
                     {"params":shared, "lr":LR_EVASION}],weight_decay=0.01)
    n_steps=len(loaders["stance_train"])*3*EPOCHS
    scheduler=get_cosine_schedule_with_warmup(optimizer,int(WARMUP_RATIO*n_steps),n_steps)

    for p in model.encoder.parameters(): p.requires_grad=False
    print(f"  Params:{sum(p.numel() for p in model.parameters()):,}  "
          f"Sampling:2:1  Scheduler:Cosine  Patience:{PATIENCE}")
    print(f"  Will save best checkpoint to: {save_path}")

    for epoch in range(1,EPOCHS+1):
        if epoch==FREEZE_EPOCHS+1:
            for p in model.encoder.parameters(): p.requires_grad=True
            print(f"  Epoch {epoch}: BERT UNFROZEN.")
        model.train()
        e_cycle=itertools.cycle(list(loaders["evasion_train"]))
        combined=[]
        for sb in loaders["stance_train"]:
            combined.append(next(e_cycle)); combined.append(next(e_cycle)); combined.append(sb)
        total=len(combined)
        run_e=run_s=ne=ns=0; t0=time.time()
        pbar = tqdm(
            combined,
            total=total,
            desc=f"Ep {epoch}/{EPOCHS}",
            leave=True
        )
        
        for step, batch in enumerate(pbar, 1):
            task = int(batch["task_id"][0].item())
            model.freeze_inactive_head(task)
        
            loss, _ = model(
                batch["input_ids"].to(device),
                batch["attention_mask"].to(device),
                batch["task_id"].to(device),
                batch["labels"].to(device),
                batch["q_input_ids"].to(device),
                batch["q_attention_mask"].to(device),
                batch["clause_ids"].to(device),
                batch["clause_masks"].to(device),
                batch["n_clauses"].to(device)
            )

            (loss / GRAD_ACCUM).backward()
        
            if task == 0:
                run_e += loss.item(); ne += 1
            else:
                run_s += loss.item(); ns += 1
        
            if step % GRAD_ACCUM == 0 or step == total:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step(); optimizer.zero_grad()
        
            # ✅ Clean update (NO postfix)
            if step % 100 == 0 or step == total:
                avg_e = run_e / ne if ne else 0
                avg_s = run_s / ns if ns else 0
                pbar.set_description(f"Ep {epoch} | E:{avg_e:.3f} S:{avg_s:.3f}")
        pbar.close()
        elapsed=(time.time()-t0)/60
        ev=evaluate(model,loaders["evasion_val"],EVASION_LABELS)
        sv=evaluate(model,loaders["stance_val"], STANCE_LABELS)
        avg=(ev["f1"]+sv["f1"])/2; pe=get_pe_f1(ev["report"]); is_best=avg>best_avg_f1
        print(f"  Ep{epoch} ({elapsed:.1f}m) E:{run_e/ne:.4f} S:{run_s/ns:.4f} | "
              f"Val E:{ev['f1']:.4f} S:{sv['f1']:.4f} Avg:{avg:.4f} PE:{pe:.2f} "
              f"{'BEST' if is_best else f'p:{patience_ctr+1}/{PATIENCE}'}")
        print("  Val per-class evasion:")
        for line in ev["report"].strip().split("\n"):
            if any(lbl in line for lbl in EVASION_LABELS):
                print(f"    {line.strip()}")
        if is_best:
            best_avg_f1=avg; patience_ctr=0; torch.save(model.state_dict(),save_path)
        else:
            patience_ctr+=1
            if patience_ctr>=PATIENCE: print(f"  Early stopping."); break
        print()

    model.load_state_dict(torch.load(save_path,map_location=device)); model.unfreeze_all()
    et=evaluate(model,loaders["evasion_test"],EVASION_LABELS)
    st=evaluate(model,loaders["stance_test"], STANCE_LABELS)
    print(f"  TEST: E:{et['f1']:.4f} S:{st['f1']:.4f} "
          f"Avg:{(et['f1']+st['f1'])/2:.4f} PE:{get_pe_f1(et['report']):.2f} "
          f"Acc:{et['acc']*100:.2f}%")
    return et,st

print("Training function ready.")

Training function ready.


In [8]:
# ─────────────────────────────────────────────────────────────
# TRAIN — V3-D (No Gating)
# ─────────────────────────────────────────────────────────────
print("\n"+"="*60)
print("  TRAINING: V3-D (No Gating) — CrossAttn + Graph + ClausePool")
print("="*60)

model = MultiTaskNoGating(ew, sw).to(device)
e_test, s_test = train_and_save(model, FULL_MODEL_PATH)

print_eval(e_test, "EVASION TEST — V3-D (No Gating)")
print_eval(s_test, "STANCE  TEST — V3-D (No Gating)")

# Save predictions
pd.DataFrame({"question":Qete,"answer":Aete,
    "true":[EVASION_LABELS[l] for l in e_test["labels"]],
    "pred":[EVASION_LABELS[p] for p in e_test["preds"]],
}).to_csv(os.path.join(OUTPUT_DIR,"preds_v3d_evasion.csv"),index=False)
pd.DataFrame({"target":Qste,"tweet":Aste,
    "true":[STANCE_LABELS[l] for l in s_test["labels"]],
    "pred":[STANCE_LABELS[p] for p in s_test["preds"]],
}).to_csv(os.path.join(OUTPUT_DIR,"preds_v3d_stance.csv"),index=False)

print(f"\nModel checkpoint saved: {FULL_MODEL_PATH}")
sz=os.path.getsize(FULL_MODEL_PATH)/(1024*1024)
print(f"  File size: {sz:.1f} MB")


  TRAINING: V3-D (No Gating) — CrossAttn + Graph + ClausePool


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

  Ep3 (36.8m) E:0.5125 S:0.4989 | Val E:0.4239 S:0.7593 Avg:0.5916 PE:0.12 BEST
  Val per-class evasion:
    Non-Evasive       0.56      0.59      0.57       154
    Partially Evasive       0.08      0.12      0.10         8
    Evasive       0.63      0.58      0.60       183



Ep 4/5:   0%|          | 0/3231 [00:00<?, ?it/s]

  Ep4 (36.8m) E:0.4904 S:0.4260 | Val E:0.4095 S:0.7474 Avg:0.5784 PE:0.12 p:1/3
  Val per-class evasion:
    Non-Evasive       0.52      0.68      0.59       154
    Partially Evasive       0.09      0.12      0.11         8
    Evasive       0.63      0.46      0.53       183



Ep 5/5:   0%|          | 0/3231 [00:00<?, ?it/s]

  Ep5 (36.8m) E:0.4833 S:0.3560 | Val E:0.4309 S:0.7592 Avg:0.5951 PE:0.12 BEST
  Val per-class evasion:
    Non-Evasive       0.55      0.59      0.57       154
    Partially Evasive       0.12      0.12      0.12         8
    Evasive       0.62      0.58      0.60       183

  TEST: E:0.4287 S:0.7630 Avg:0.5958 PE:0.12 Acc:57.68%

  EVASION TEST — V3-D (No Gating)
  Loss:1.5789  Acc:57.68%  MacroF1:0.4287  Prec:0.4265  Rec:0.4325

  Per-class Report:
  precision    recall  f1-score   support
  
        Non-Evasive       0.54      0.57      0.56       154
  Partially Evasive       0.10      0.12      0.11         8
            Evasive       0.64      0.60      0.62       183
  
           accuracy                           0.58       345
          macro avg       0.43      0.43      0.43       345
       weighted avg       0.58      0.58      0.58       345

  Confusion Matrix:
                        Pred:Non-Evasive  Pred:Partially Evasive  Pred:Evasive
True:Non-Evasive            

In [9]:
# ─────────────────────────────────────────────────────────────
# TASK 3 — EXPLANATION SPAN EXTRACTION
#
# How it works:
# The No-Gating architecture has two natural explanation signals:
#
# 1. cross_attn.last_attn_weights [B, n_clauses]
#    — which clauses the question "attended to" most
#    — high weight = this clause is relevant to the question
#    — for evasive answers: high weight + evasive prediction = this
#      clause is where the evasion happens
#
# 2. cl_pool.last_clause_weights [B, n_clauses]
#    — which clauses the pooling layer weighted most after graph reasoning
#    — captures structural importance independent of the question
#
# We combine both signals: final_importance = 0.6*cross_attn + 0.4*cl_pool
# Then highlight the top-weighted clause(s) in the answer.
# ─────────────────────────────────────────────────────────────

@torch.no_grad()
def extract_explanation(model, question, answer):
    """
    Run a single Q+A pair through the model and extract:
    - predicted evasion label
    - predicted stance label
    - explanation: the most important clause(s) with importance scores
    """
    model.eval()

    # Encode
    enc  = tokenizer(question,answer,truncation=True,padding="max_length",
                     max_length=MAX_LEN,return_tensors="pt").to(device)
    qenc = tokenizer(question,truncation=True,padding="max_length",
                     max_length=Q_MAX_LEN,return_tensors="pt").to(device)

    # Encode clauses
    clauses = split_into_clauses(answer)
    ids_l,masks_l=[],[]
    for c in clauses[:MAX_CLAUSES]:
        ce=tokenizer(question,c,truncation=True,padding="max_length",
                     max_length=CLAUSE_LEN,return_tensors="pt").to(device)
        ids_l.append(ce["input_ids"])
        masks_l.append(ce["attention_mask"])
    n_real=len(ids_l)
    while len(ids_l)<MAX_CLAUSES:
        ids_l.append(torch.zeros(1,CLAUSE_LEN,dtype=torch.long,device=device))
        masks_l.append(torch.zeros(1,CLAUSE_LEN,dtype=torch.long,device=device))
    clause_ids   = torch.cat(ids_l,dim=0).unsqueeze(0)
    clause_masks = torch.cat(masks_l,dim=0).unsqueeze(0)
    n_clauses    = torch.tensor([n_real],dtype=torch.long,device=device)

    # Evasion prediction
    _,e_logits=model(enc["input_ids"],enc["attention_mask"],
                     torch.tensor([0],device=device),
                     q_input_ids=qenc["input_ids"],
                     q_attention_mask=qenc["attention_mask"],
                     clause_ids=clause_ids,clause_masks=clause_masks,
                     n_clauses=n_clauses)
    evasion_pred = EVASION_LABELS[torch.argmax(e_logits,dim=1).item()]
    evasion_conf = float(torch.softmax(e_logits,dim=1).max())

    # Stance prediction
    enc_s  = tokenizer(question,answer,truncation=True,padding="max_length",
                       max_length=STANCE_MAX_LEN,return_tensors="pt").to(device)
    qenc_s = tokenizer(question,truncation=True,padding="max_length",
                       max_length=Q_MAX_LEN,return_tensors="pt").to(device)
    # For stance we still need clause tensors (model expects them)
    _,s_logits=model(enc_s["input_ids"],enc_s["attention_mask"],
                     torch.tensor([1],device=device),
                     q_input_ids=qenc_s["input_ids"],
                     q_attention_mask=qenc_s["attention_mask"],
                     clause_ids=clause_ids,clause_masks=clause_masks,
                     n_clauses=n_clauses)
    stance_pred = STANCE_LABELS[torch.argmax(s_logits,dim=1).item()]
    stance_conf = float(torch.softmax(s_logits,dim=1).max())

    # Explanation: combine cross-attention + clause-pool weights
    ca_weights = model.cross_attn.last_attn_weights[0,:n_real].cpu().numpy()
    cp_weights = model.cl_pool.last_clause_weights[0,:n_real].cpu().numpy()
    # Normalise
    ca_weights = ca_weights / (ca_weights.sum()+1e-9)
    cp_weights = cp_weights / (cp_weights.sum()+1e-9)
    importance = 0.6*ca_weights + 0.4*cp_weights

    clause_scores = [(clauses[i], float(importance[i])) for i in range(n_real)]
    clause_scores.sort(key=lambda x: x[1], reverse=True)

    return {
        "evasion_pred":  evasion_pred,
        "evasion_conf":  evasion_conf,
        "stance_pred":   stance_pred,
        "stance_conf":   stance_conf,
        "clause_scores": clause_scores,
        "top_clause":    clause_scores[0][0] if clause_scores else answer[:200],
    }

def visualize_explanation(question, answer, result):
    """Render HTML heatmap showing clause importance."""
    clauses_text = split_into_clauses(answer)
    n = len(clauses_text)
    ca_w = model.cross_attn.last_attn_weights[0,:n].cpu().numpy()
    cp_w = model.cl_pool.last_clause_weights[0,:n].cpu().numpy()
    ca_w = ca_w/(ca_w.sum()+1e-9); cp_w = cp_w/(cp_w.sum()+1e-9)
    importance = 0.6*ca_w + 0.4*cp_w

    evasion_color = {"Non-Evasive":"#4CAF50","Partially Evasive":"#FF9800","Evasive":"#F44336"}
    stance_color  = {"FAVOR":"#2196F3","AGAINST":"#9C27B0"}

    ec = evasion_color.get(result["evasion_pred"],"#999")
    sc = stance_color.get(result["stance_pred"],"#999")
    html = (
        "<div style='font-family:sans-serif;max-width:800px;padding:16px;"
        "border:1px solid #ddd;border-radius:8px;'>"
        f"<p style='font-size:13px;color:#666;margin:0 0 8px'><b>Question:</b> {question[:200]}</p>"
        "<div style='display:flex;gap:12px;margin-bottom:12px;'>"
        f"<span style='background:{ec};color:white;padding:4px 10px;"
        f"border-radius:4px;font-size:13px;font-weight:500;'>"
        f"{result['evasion_pred']} ({result['evasion_conf']*100:.0f}%)</span>"
        f"<span style='background:{sc};color:white;padding:4px 10px;"
        f"border-radius:4px;font-size:13px;font-weight:500;'>"
        f"{result['stance_pred']} ({result['stance_conf']*100:.0f}%)</span>"
        "</div>"
        "<p style='font-size:12px;color:#888;margin:0 0 8px'>"
        "Answer clauses — darker highlight = higher evasion relevance</p>"
        "<div style='line-height:1.9;font-size:14px;'>"
    )
    for i,clause in enumerate(clauses_text):
        imp = float(importance[i]) if i<len(importance) else 0.0
        alpha = min(0.85, imp*3.0)
        html += f"<span style='background:rgba(244,67,54,{alpha:.2f});padding:2px 4px;border-radius:3px;margin:2px;display:inline;'>{clause}</span> "
    html += "</div>"
    if result["evasion_pred"] != "Non-Evasive":
        html += f"<p style='font-size:12px;color:#666;margin:8px 0 0'><b>Most relevant clause:</b> {result['top_clause'][:150]}...</p>"
        html += "</div>"
    display(HTML(html))

print("Task 3 — Explanation extraction functions ready.")
print("  extract_explanation(model, question, answer) -> dict with all 3 task outputs")
print("  visualize_explanation(question, answer, result) -> HTML heatmap")

Task 3 — Explanation extraction functions ready.
  extract_explanation(model, question, answer) -> dict with all 3 task outputs
  visualize_explanation(question, answer, result) -> HTML heatmap


In [10]:
# ─────────────────────────────────────────────────────────────
# RUN TASK 3 — Qualitative case studies
# Show examples from each evasion class with highlighted clauses
# ─────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("  TASK 3 — EXPLANATION SPAN EXTRACTION (Qualitative Analysis)")
print("="*60)

# Pick 2-3 examples from each class in the test set
for target_label in ["Non-Evasive","Partially Evasive","Evasive"]:
    class_id = EVASION_LABEL2ID[target_label]
    # Get test indices for this class that were correctly predicted
    correct_idx = [i for i,(t,p) in enumerate(zip(e_test["labels"],e_test["preds"]))
                   if t==class_id and p==class_id]
    wrong_idx   = [i for i,(t,p) in enumerate(zip(e_test["labels"],e_test["preds"]))
                   if t==class_id and p!=class_id]

    print(f"\n{'─'*50}")
    print(f"  Class: {target_label}  (correct:{len(correct_idx)}  wrong:{wrong_idx[:3]})")
    print(f"{'─'*50}")

    # Show up to 2 correct predictions
    for k,idx in enumerate(correct_idx[:2]):
        q = Qete[idx]; a = Aete[idx]
        result = extract_explanation(model, q, a)
        print(f"\nExample {k+1} (CORRECT prediction):")
        visualize_explanation(q, a, result)

    # Show 1 wrong prediction if any
    if wrong_idx:
        idx = wrong_idx[0]
        q = Qete[idx]; a = Aete[idx]
        result = extract_explanation(model, q, a)
        true_lbl = EVASION_LABELS[e_test["labels"][idx]]
        result["true_label"] = true_lbl
        print(f"\nExample (WRONG prediction — true: {true_lbl}):")
        visualize_explanation(q, a, result)

# Save explanation results for all test samples
print("\nGenerating explanations for all test samples...")
explanation_rows = []
for i,(q,a) in enumerate(zip(Qete,Aete)):
    try:
        res = extract_explanation(model, q, a)
        explanation_rows.append({
            "question":         q,
            "answer":           a,
            "true_evasion":     EVASION_LABELS[e_test["labels"][i]],
            "pred_evasion":     res["evasion_pred"],
            "evasion_conf":     round(res["evasion_conf"],4),
            "pred_stance":      res["stance_pred"],
            "stance_conf":      round(res["stance_conf"],4),
            "top_clause":       res["top_clause"],
            "clause_scores":    str([(c[:80],round(s,4)) for c,s in res["clause_scores"]]),
        })
    except Exception as e:
        pass

df_expl = pd.DataFrame(explanation_rows)
expl_path = os.path.join(OUTPUT_DIR,"task3_explanations.csv")
df_expl.to_csv(expl_path,index=False)
print(f"  Saved {len(df_expl)} explanation records to {expl_path}")


  TASK 3 — EXPLANATION SPAN EXTRACTION (Qualitative Analysis)

──────────────────────────────────────────────────
  Class: Non-Evasive  (correct:88  wrong:[0, 4, 5])
──────────────────────────────────────────────────

Example 1 (CORRECT prediction):



Example 2 (CORRECT prediction):



Example (WRONG prediction — true: Non-Evasive):



──────────────────────────────────────────────────
  Class: Partially Evasive  (correct:1  wrong:[114, 196, 226])
──────────────────────────────────────────────────

Example 1 (CORRECT prediction):



Example (WRONG prediction — true: Partially Evasive):



──────────────────────────────────────────────────
  Class: Evasive  (correct:110  wrong:[1, 2, 3])
──────────────────────────────────────────────────

Example 1 (CORRECT prediction):



Example 2 (CORRECT prediction):



Example (WRONG prediction — true: Evasive):



Generating explanations for all test samples...
  Saved 345 explanation records to /kaggle/working/task3_explanations.csv


In [11]:
# ─────────────────────────────────────────────────────────────
# FINAL RESULTS TABLE
# ─────────────────────────────────────────────────────────────
print("\n"+"="*75)
print("  FINAL RESULTS — ALL SUBMISSIONS")
print("="*75)

results = {
    "Mid V1 (baseline)":          {"e":0.4592,"s":0.7651,"avg":0.6122,"pe":0.14},
    "Mid V2 (gating)":            {"e":0.4976,"s":0.7633,"avg":0.6304,"pe":0.27},
    "Mid V3-Full (no aug)":       {"e":0.4692,"s":0.7592,"avg":0.6142,"pe":0.13},
    "Ablation V3-C (gating only)":{"e":0.5055,"s":0.7669,"avg":0.6362,"pe":0.27},
    "Ablation V3-A (no graph)":   {"e":0.4462,"s":0.7694,"avg":0.6078,"pe":0.13},
    "Ablation V3-D (no gating)":  {"e":0.4495,"s":0.7524,"avg":0.6010,"pe":0.10},
    "FINAL V3-D (this run)":      {"e":e_test["f1"],"s":s_test["f1"],
                                   "avg":(e_test["f1"]+s_test["f1"])/2,
                                   "pe":get_pe_f1(e_test["report"])},
}

best_avg = max(v["avg"] for v in results.values())
print(f"  {'Variant':<42} {'E.F1':>7} {'S.F1':>7} {'Avg':>7} {'PE':>6}")
print(f"  {'─'*68}")
for name,r in results.items():
    mark=" <- BEST" if abs(r["avg"]-best_avg)<1e-4 else ""
    print(f"  {name:<42} {r['e']:>7.4f} {r['s']:>7.4f} {r['avg']:>7.4f} {r['pe']:>6.2f}{mark}")
print(f"  {'='*68}")

pd.DataFrame([{"variant":k,"e_f1":v["e"],"s_f1":v["s"],"avg_f1":v["avg"],"pe_f1":v["pe"]}
              for k,v in results.items()
]).to_csv(os.path.join(OUTPUT_DIR,"final_results_all.csv"),index=False)

print(f"\nSaved files in {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    sz=os.path.getsize(os.path.join(OUTPUT_DIR,f))
    unit="MB" if sz>100000 else "KB"
    val=sz/(1024*1024) if sz>100000 else sz/1024
    print(f"  {f:<45} {val:.1f} {unit}")


  FINAL RESULTS — ALL SUBMISSIONS
  Variant                                       E.F1    S.F1     Avg     PE
  ────────────────────────────────────────────────────────────────────
  Mid V1 (baseline)                           0.4592  0.7651  0.6122   0.14
  Mid V2 (gating)                             0.4976  0.7633  0.6304   0.27
  Mid V3-Full (no aug)                        0.4692  0.7592  0.6142   0.13
  Ablation V3-C (gating only)                 0.5055  0.7669  0.6362   0.27 <- BEST
  Ablation V3-A (no graph)                    0.4462  0.7694  0.6078   0.13
  Ablation V3-D (no gating)                   0.4495  0.7524  0.6010   0.10
  FINAL V3-D (this run)                       0.4287  0.7630  0.5958   0.12

Saved files in /kaggle/working:
  .virtual_documents                            4.0 KB
  best_model_full.pt                            275.0 MB
  final_results_all.csv                         0.4 KB
  preds_v3d_evasion.csv                         0.7 MB
  preds_v3d_stance.csv 

In [12]:
# ═══════════════════════════════════════════════════════════
# INFERENCE — Load saved model and predict on new examples
# Run this cell independently after training is complete
# ═══════════════════════════════════════════════════════════

def load_model_for_inference(model_path, device):
    """Load saved model checkpoint for inference."""
    # Rebuild class weights (needed for model constructor)
    # In practice you would save/load these too, but here we just use
    # uniform weights since they don't affect inference
    dummy_ew = torch.ones(3).to(device)
    dummy_sw = torch.ones(2).to(device)
    m = MultiTaskNoGating(dummy_ew, dummy_sw).to(device)
    m.load_state_dict(torch.load(model_path, map_location=device))
    m.eval()
    print(f"Model loaded from {model_path}")
    print(f"  Params: {sum(p.numel() for p in m.parameters()):,}")
    return m

def predict(model, question, answer, target=None):
    """
    Run inference on a single Q+A pair.
    Returns evasion label, stance label, and explanation.

    Args:
        question : str — the interview question
        answer   : str — the politician's answer
        target   : str — stance target (e.g. 'Donald Trump'), optional

    Returns:
        dict with keys: evasion_pred, evasion_conf, stance_pred,
                        stance_conf, top_clause, clause_scores
    """
    # Use the answer as target if not provided
    if target is None:
        target = question  # question acts as target for stance too
    result = extract_explanation(model, question, answer)
    return result

def predict_batch(model, questions, answers):
    """Run inference on a list of Q+A pairs."""
    results = []
    for q,a in tqdm(zip(questions,answers),total=len(questions),desc="Predicting"):
        try:
            results.append(predict(model,q,a))
        except Exception as e:
            results.append({"evasion_pred":"Error","stance_pred":"Error","error":str(e)})
    return results

# ── DEMO: Load model and predict on 5 new examples ────────────
print("Loading saved model for inference...")
inference_model = load_model_for_inference(FULL_MODEL_PATH, device)

# Use a few val examples as demo
demo_questions = Qeva[:5]
demo_answers   = Aeva[:5]
demo_true      = [EVASION_LABELS[y] for y in yeva[:5]]

print("\nRunning inference on 5 examples:")
print("─"*60)
for i,(q,a,true_lbl) in enumerate(zip(demo_questions,demo_answers,demo_true)):
    result = predict(inference_model, q, a)
    correct = "✓" if result["evasion_pred"]==true_lbl else "✗"
    print(f"\nExample {i+1} {correct}")
    print(f"  Q: {q[:100]}...")
    print(f"  True:  {true_lbl}")
    print(f"  Pred:  {result['evasion_pred']} ({result['evasion_conf']*100:.0f}%) | "
          f"Stance: {result['stance_pred']} ({result['stance_conf']*100:.0f}%)")
    print(f"  Key clause: {result['top_clause'][:120]}")

print("\n" + "─"*60)
print("Inference complete.")
print("\nTo load and use in a separate notebook:")
print("""
  from transformers import DistilBertTokenizer
  # (copy model class definition here)
  model = load_model_for_inference('/kaggle/working/best_model_full.pt', device)
  result = predict(model, question, answer)
  print(result['evasion_pred'], result['stance_pred'])
""")

Loading saved model for inference...


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded from /kaggle/working/best_model_full.pt
  Params: 72,077,062

Running inference on 5 examples:
────────────────────────────────────────────────────────────

Example 1 ✗
  Q: Q. Now, back home, when you reflect upon your tour of India, the first diplomatic visit to the count...
  True:  Non-Evasive
  Pred:  Evasive (93%) | Stance: AGAINST (62%)
  Key clause: The President.

Example 2 ✗
  Q: Q. And his job is safe?...
  True:  Non-Evasive
  Pred:  Evasive (69%) | Stance: FAVOR (62%)
  Key clause: Dissemination of Information Regarding Oil Spill

Example 3 ✗
  Q: Q. Two weeks ago, Mr. President, your Acting OMB Director was in this room and was talking about wha...
  True:  Evasive
  Pred:  Non-Evasive (86%) | Stance: FAVOR (86%)
  Key clause: In fact, the administration has the—as you know, the lowest average unemployment of any administration in history.

Example 4 ✓
  Q: Q. Your successor spoke by phone with the President of Taiwan the other day and declared subsequentl...